# WLASL300 Sign Language Recognition — Training Notebook

End-to-end training pipeline on Google Colab.

**Prerequisites:**
1. Enable GPU: `Runtime → Change runtime type → T4 GPU`
2. Mount Google Drive (cell below) — used to persist checkpoints across sessions.
3. Upload the Kaggle credentials JSON or set `KAGGLE_USERNAME` / `KAGGLE_KEY` secrets.

**Sections:**
1. Environment setup
2. Mount Google Drive
3. Download dataset
4. Build annotations
5. Training
6. Evaluation and plots

## 1. Environment setup

In [ ]:
# Verify GPU is available
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Clone the project repository
# Replace with your actual repository URL
!git clone https://github.com/your-username/wlasl300-sign-language.git
%cd wlasl300-sign-language

In [ ]:
# Install dependencies (uv is not available on Colab — use pip)
!pip install -q \
    torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 \
    decord opencv-python gensim \
    scikit-learn pandas tqdm pillow \
    matplotlib seaborn plotly \
    wandb pyyaml dacite loguru rich einops \
    pytorchvideo

## 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Set this to your Drive folder for persisting data and checkpoints
DRIVE_DIR = "/content/drive/MyDrive/wlasl300"

import os

os.makedirs(f"{DRIVE_DIR}/dataset/raw", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/trained_models/best", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/trained_models/latest", exist_ok=True)

# Symlink Drive dirs into the project so paths in config.yaml work
!ln -sfn {DRIVE_DIR}/dataset/raw data/raw
!ln -sfn {DRIVE_DIR}/trained_models/best trained_models/best
!ln -sfn {DRIVE_DIR}/trained_models/latest trained_models/latest
print("Drive mounted and symlinks created.")

## 3. Download dataset from Kaggle

In [ ]:
# Option A: Upload kaggle.json via Colab file picker
from google.colab import files

# Uncomment to upload:
# uploaded = files.upload()  # select your kaggle.json

# Option B: Set credentials directly
import os

os.environ["KAGGLE_USERNAME"] = "your_kaggle_username"  # REPLACE
os.environ["KAGGLE_KEY"] = "your_kaggle_api_key"  # REPLACE

!mkdir -p ~/.kaggle
!echo '{"username":"'$KAGGLE_USERNAME'","key":"'$KAGGLE_KEY'"}' > ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Download WLASL300 dataset (~2 GB)
!kaggle datasets download vodinhnhattruong/dataset-wlasl300 \
    --unzip --path data/raw/

# Download the official WLASL annotation JSON
!wget -q https://raw.githubusercontent.com/dxli94/WLASL/master/data/WLASL_v0.3.json \
    -O data/raw/WLASL_v0.3.json

print("Dataset downloaded.")
!ls data/raw/ | head -20

## 4. Build annotations and Word2Vec embeddings

In [ ]:
# Download Google News Word2Vec binary (~1.5 GB)
# This uses gdown to fetch from Google Drive
!pip install -q gdown
!gdown -O /content/GoogleNews-vectors-negative300.bin.gz
!gunzip -f /content/GoogleNews-vectors-negative300.bin.gz
print("Word2Vec model downloaded.")

In [ ]:
!python dataset/annotations/build_annotations.py \
    --raw_dir            data/raw \
    --preprocessing_dir  preprocessing \
    --wlasl_dir          WLASL300 \
    --folder2label       folder2label_str.txt \
    --word2vec_bin       trained_models/embeddings/GoogleNews-vectors-negative300.bin \
    --out_dir            dataset/annotations

## 5. Training

In [ ]:
# Start training (uses config/config.yaml)
# Reduce batch_size if you hit CUDA OOM
!python train/train.py \
    --config config/config.yaml

In [ ]:
# To resume from a checkpoint:
# !python train/train.py \
#     --config config/config.yaml \
#     --resume trained_models/latest/checkpoint_epoch010.pt

## 6. Evaluation and visualisation

In [ ]:
from IPython.display import Image, display
import os

plot_dir = "logs/plots"
for fname in ["loss_curves.png", "accuracy_curves.png", "throughput.png"]:
    path = os.path.join(plot_dir, fname)
    if os.path.exists(path):
        print(f"--- {fname} ---")
        display(Image(path))

In [ ]:
# Run inference on a sample video
# Replace with the path to a real video file
SAMPLE_VIDEO = "WLASL300/0/00412.mp4"

!python inference/inference.py \
    --checkpoint trained_models/best/checkpoint.pt \
    --video      {SAMPLE_VIDEO} \
    --top_k      5 \
    --verbose